# RSNA Knee — Modeling v21: Anatomy-Query Head (k-fold + laterality)

**Referensi**: `prvsiyan/head-and-shoulders-knees-and-toes` (V19, 20 Agustus 2026) — anatomy-query
specialist tervalidasi **+0,00989 macro AUC** (0,84199 → 0,85188 di OOF gold-58, P(gain>0)=0.998).
Arsitekturnya: **12 finding-query cross-attend** ke 24 slice-token anatomis (8 sagittal + 8 coronal +
8 axial), tiap token = triplet RGB slice fisik berdekatan, dgn **plane embedding + posisi kedalaman**,
plus laterality canonicalization (orientasi kanonik lutut kanan).

**v21 ini = v20 (k-fold, cache preprocessing v4/laterality) TAPI head diganti** dari SlotHead
(per-slot attention pooling, arsitektur kita sejak v6) ke **AnatomyQueryHead** (12 query cross-attention
atas 24 token anatomis). Perbedaan struktural TUNGGAL dari v20 adalah model/head + dataset sampling;
semua hyperparameter training, k-fold, loss, scheduler SAMA PERSIS dgn v20/v18.

**Pembanding metrik (semua OOF 5-fold, apples-to-apples)**:
- v18 OOF (cache v2, SlotHead) = **0,8068**
- v20 OOF (cache v4, SlotHead) = belum jalan
- **v21 OOF (cache v4, AnatomyQueryHead) = TARGET eksperimen ini**

Hipotesis: kalau v21 OOF > v18/v20 melebihi noise antar-fold (std ±0,034), anatomy-query head adalah
pengungkit nyata — checkpoint v21 jadi model final baru. Kalau tidak, SlotHead tidak jadi bottleneck.

**Catatan RGB triplet**: v16 (channel RGB genuine dari 3 slice jauh) GAGAL krn patch embedding beku.
Di sini triplet = 3 slice **berdekatan fisik** (tetangga ±1 dari anchor, ~nyaris identik) — pola
channel lebih dekat ke grayscale-dengan-tekstur-lokal drpd v16, dan bakal diuji k-fold utk kepastian.

**Estimasi waktu**: ~4,5-5 jam (sama spt v18/v20; per-studdy 24 token vs 24 gambar v15).

## 0. Setup

In [ ]:
import os, sys, json, time, random, traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

ensure("transformers")
ensure("scikit-learn", "sklearn")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

XLA_OK = False
xm = None
try:
    import torch_xla.core.xla_model as xm
    _xla_dev = xm.xla_device()
    _ = torch.zeros(1, device=_xla_dev) + 1
    XLA_OK = True
except Exception as e:
    print(f"[info] torch_xla tidak tersedia/tidak usable ({e}) -- fallback cek CUDA/CPU.")

def _cuda_usable():
    if not torch.cuda.is_available():
        return False
    try:
        _ = torch.zeros(1, device="cuda") + 1
        return True
    except Exception as e:
        print(f"[warn] GPU terdeteksi tapi tidak kompatibel ({e}) -- fallback CPU.")
        return False

if XLA_OK:
    DEVICE = _xla_dev
elif _cuda_usable():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
tlog(f"device: {DEVICE} | xla: {XLA_OK} | gpu_count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

OUT_DIR = Path("model_outputs")
OUT_DIR.mkdir(exist_ok=True)


## 1. Cari sumber data (preprocessing cache, weak labels, DINOv2 weights)

In [ ]:
kaggle_input = Path("/kaggle/input")

def find_dir_containing(filename, hint=""):
    if not kaggle_input.exists():
        return None
    candidates = []
    for root, dirs, files in os.walk(kaggle_input):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if filename in files:
            candidates.append(Path(root))
    if not candidates:
        return None
    if hint:
        for c in candidates:
            if hint.lower() in str(c).lower():
                return c
    return candidates[0]

PREP_DIR = find_dir_containing("series_manifest.csv", "preprocessing")
LABELER_DIR = find_dir_containing("weak_labels_full.csv", "labeler")
LLM_DIR = find_dir_containing("report_labels_v4hybrid.csv", "hybrid-report-labels")
tlog(f"PREP_DIR: {PREP_DIR}")
tlog(f"LABELER_DIR: {LABELER_DIR}")
tlog(f"LLM_DIR: {LLM_DIR}")

def find_dinov2(variant="base"):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None

DINOV2_DIR = find_dinov2("base")
tlog(f"DINOV2_DIR: {DINOV2_DIR}")

CKPT_DIR = find_dir_containing("best_model.pt", "modeling")
tlog(f"CKPT_DIR: {CKPT_DIR}")

assert PREP_DIR is not None, "preprocessing kernel belum di-attach (kernel_sources)"
assert LABELER_DIR is not None, "weak-labeler kernel belum di-attach (kernel_sources)"
assert LLM_DIR is not None, "dataset public hybrid-report-labels belum di-attach (dataset_sources)"
assert DINOV2_DIR is not None, "DINOv2 model belum di-attach (model_sources)"

## 2. Load & gabungkan: label, manifest series, cache path

Sumber label: `flight0234/rsna-knee-hybrid-report-labels` (0.8991 vs gold, dipakai sejak v8) utk 4.349
studi non-gold; 57 studi gold TETAP pakai label asli annotator, tidak pernah ditimpa. Studi gold yang
ter-flag anomali overexposure dari audit exhaustive di-exclude.

In [ ]:
series_manifest = pd.read_csv(PREP_DIR / "series_manifest.csv")
study_manifest = pd.read_csv(PREP_DIR / "study_manifest.csv")
weak_labels = pd.read_csv(LABELER_DIR / "weak_labels_full.csv")
llm_labels = pd.read_csv(LLM_DIR / "report_labels_v4hybrid.csv")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

tlog(f"series_manifest: {series_manifest.shape} | study_manifest: {study_manifest.shape} "
     f"| weak_labels: {weak_labels.shape} | llm_labels: {llm_labels.shape}")

EXCLUDE_STUDIES = {"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921"}
weak_labels = weak_labels[~weak_labels["StudyInstanceUID"].isin(EXCLUDE_STUDIES)].reset_index(drop=True)
series_manifest = series_manifest[series_manifest["status"] == "OK"].reset_index(drop=True)

gold_mask = weak_labels["source"] == "gold"
CONF_COLS = [c + "__conf" for c in LABEL_COLS]
train_labels = llm_labels[["StudyInstanceUID"] + LABEL_COLS + CONF_COLS].merge(
    weak_labels[["StudyInstanceUID", "lang"]], on="StudyInstanceUID", how="right")
train_labels = train_labels.set_index("StudyInstanceUID")
gold_rows = weak_labels.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].set_index("StudyInstanceUID")
train_labels.loc[gold_rows.index, LABEL_COLS] = gold_rows[LABEL_COLS]
train_labels["source"] = np.where(train_labels.index.isin(gold_rows.index), "gold", "flight0234_v4hybrid")
for c in CONF_COLS:
    train_labels.loc[gold_rows.index, c] = 1.0
train_labels = train_labels.reset_index()

tlog(f"Setelah exclude anomali & gabung label: {train_labels.shape[0]} studi label")
tlog(f"Studi gold bersih: {(train_labels['source']=='gold').sum()}")


## 3. Pilih 1 series terbaik per plane (bukan slot bag) + indeks cache

Anatomy-query (resep prvsiyan V19) butuh **1 seri representatif per plane** (Sagittal/Coronal/Axial),
bukan 6-slot bag. Kriteria pilih: prioritas fluid-sensitive + fat-suppression (scoring sama dgn
prvsiyan: `4*Fluid_Sensitive + 3*Fat_Suppression`). Kalau plane tidak ada, token-nya di-mask 0.

In [ ]:
PLANES = ["Sagittal", "Coronal", "Axial"]
N_PER_PLANE = 8       # token per plane -> total 24 token/studi (8x3)
CACHE_SLICES = 16     # preprocessing v4 cache: 16 slice/series
IMG_SIZE = 224

def truthy(v):
    return str(v).strip().lower() in {"1", "true", "t", "yes", "y", "on"}

series_manifest["plane"] = series_manifest["Anatomical_Plane"].str.strip().str.title()
series_manifest["pick_score"] = (
    4 * series_manifest["Fluid_Sensitive"].map(truthy).astype(int)
    + 3 * series_manifest["Fat_Suppression"].map(truthy).astype(int)
)
picked = (
    series_manifest[series_manifest["plane"].isin(PLANES)]
    .sort_values(["StudyInstanceUID", "plane", "pick_score", "SeriesInstanceUID"],
                 ascending=[True, True, False, True])
    .groupby(["StudyInstanceUID", "plane"], as_index=False)
    .first()
)
print(picked.columns.tolist())
tlog(f"Picked series per (studi, plane): {len(picked)} baris")
print(picked.groupby("plane")["SeriesInstanceUID"].count())
print(picked["laterality_final"].value_counts(dropna=False) if "laterality_final" in picked.columns else "(kolom laterality_final tidak ada di picked)")

## 4. PyTorch Dataset: 1 studi -> 24 token anatomis (8 per plane), tiap token = triplet RGB

In [ ]:
# indeks: (study, plane) -> {cache_path, laterality}
study_plane_cache = {}
n_missing_file = 0
for _, r in picked.iterrows():
    p = PREP_DIR / "image_cache" / str(r["StudyInstanceUID"]) / f'{r["SeriesInstanceUID"]}.npz'
    if p.exists():
        study_plane_cache[(str(r["StudyInstanceUID"]), r["plane"])] = {
            "path": p,
            "laterality": r.get("laterality_final"),
        }
    else:
        n_missing_file += 1
tlog(f"study_plane_cache: {len(study_plane_cache)} (studi,plane) terindeks | {n_missing_file} file cache tidak ketemu")

N_TOKEN = len(PLANES) * N_PER_PLANE  # 24

class AnatomyDataset(Dataset):
    """1 studi -> N_TOKEN token. Tiap token = triplet RGB (tetangga -1/0/+1) dari 1 anchor slice."""
    def __init__(self, study_ids, labels_df):
        self.study_ids = list(study_ids)
        self.labels = labels_df.set_index("StudyInstanceUID")

    def __len__(self):
        return len(self.study_ids)

    def _load_volume(self, study, plane):
        info = study_plane_cache.get((study, plane))
        if info is None:
            return None
        try:
            vol = np.load(info["path"])["slices"]  # (16,224,224) uint8, urutan fisik v2
        except Exception:
            return None
        # laterality canonicalization utk sagittal: kalau lutut KIRI, balik urutan stack
        # supaya token index 0 = paling medial (resep prvsiyan V19). Coronal/Axial sudah
        # di-flip horizontal di preprocessing v4.
        if plane == "Sagittal" and str(info["laterality"]).upper() == "L":
            vol = vol[::-1]
        return vol

    def _plane_tokens(self, vol):
        """16 slice -> N_PER_PLANE anchor merata, tiap anchor triplet (i-1,i,i+1)."""
        n = vol.shape[0]
        if n >= 3:
            centers = np.linspace(1, n - 2, N_PER_PLANE).round().astype(int)
        else:
            centers = np.zeros(N_PER_PLANE, dtype=int)
        # triplet (3,224,224) per anchor -> (N_PER_PLANE, 3, 224, 224)
        triplets = np.stack([vol[centers - 1], vol[centers], vol[centers + 1]], axis=1)
        return triplets.astype(np.float32) / 255.0

    def __getitem__(self, idx):
        study = self.study_ids[idx]
        imgs = np.zeros((N_TOKEN, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        mask = np.zeros(N_TOKEN, dtype=np.float32)
        plane_id = np.zeros(N_TOKEN, dtype=np.int64)
        position = np.zeros(N_TOKEN, dtype=np.float32)
        for pi, plane in enumerate(PLANES):
            vol = self._load_volume(study, plane)
            if vol is not None:
                start = pi * N_PER_PLANE
                imgs[start:start + N_PER_PLANE] = self._plane_tokens(vol)
                mask[start:start + N_PER_PLANE] = 1.0
                plane_id[start:start + N_PER_PLANE] = pi
                position[start:start + N_PER_PLANE] = np.linspace(0, 1, N_PER_PLANE)

        row = self.labels.loc[study] if study in self.labels.index else None
        target = np.full(len(LABEL_COLS), np.nan, dtype=np.float32)
        weight = np.zeros(len(LABEL_COLS), dtype=np.float32)
        if row is not None:
            for i, c in enumerate(LABEL_COLS):
                v = row.get(c)
                if pd.notna(v):
                    target[i] = float(v)
                    cv = row.get(c + "__conf")
                    weight[i] = float(cv) if pd.notna(cv) else 1.0

        return {"study": study,
                "imgs": torch.from_numpy(np.ascontiguousarray(imgs)),  # (N_TOKEN,3,H,W)
                "mask": torch.from_numpy(mask),            # (N_TOKEN,)
                "plane_id": torch.from_numpy(plane_id),    # (N_TOKEN,)
                "position": torch.from_numpy(position),    # (N_TOKEN,) 0..1 dalam plane
                "target": torch.from_numpy(target),
                "weight": torch.from_numpy(weight)}

## 5. Arsitektur: AnatomyQueryHead (12 query cross-attention atas 24 token)

Resep persis prvsiyan V19: tiap token = embedding backbone (CLS + mean-patch concat), ditambah
plane embedding + posisi kedalaman. 12 query (satu per finding) cross-attend ke SEMUA token (dgn
mask), lalu hasilnya dikalikan bobot per-finding + bias -> logits 12 label.

In [ ]:
import math


class AnatomyQueryHead(nn.Module):
    """Cross-attention: 12 query (per finding) attend ke N_TOKEN slice-token."""
    def __init__(self, d_in, d=256, n_target=12, n_plane=3, dropout=0.15):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(d_in), nn.Linear(d_in, d), nn.GELU(), nn.Dropout(dropout),
        )
        self.plane = nn.Embedding(n_plane, d)
        self.pos = nn.Sequential(nn.Linear(1, d), nn.Tanh())
        self.query = nn.Parameter(torch.empty(n_target, d))
        self.cls_w = nn.Parameter(torch.empty(n_target, d))
        self.cls_b = nn.Parameter(torch.zeros(n_target))
        nn.init.normal_(self.query, std=0.02)
        nn.init.normal_(self.cls_w, std=0.02)
        self.d = d

    def forward(self, features, mask, plane_id, position):
        # features (B, T, d_in); mask (B, T); plane_id (B, T); position (B, T)
        hidden = (
            self.proj(features) + self.plane(plane_id)
            + self.pos(position.unsqueeze(-1))
        )
        score = torch.einsum("bsd,td->bts", hidden, self.query) / math.sqrt(self.d)
        score = score.masked_fill(mask.unsqueeze(1) < 0.5, -1e4)
        att = score.softmax(-1)
        context = torch.einsum("bts,bsd->btd", att, hidden)
        return (context * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b.unsqueeze(0)


class AnatomyModel(nn.Module):
    def __init__(self, backbone_path, unfreeze_last=6):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(str(backbone_path))
        if DEVICE.type == 'xla':
            print("[info] grad checkpointing DINONAKTIFKAN di device XLA (v13).")
        else:
            try:
                self.backbone.gradient_checkpointing_enable()
            except Exception as e:
                print(f"[warn] gradient_checkpointing_enable() gagal ({e}) -- lanjut TANPA.")
        n_layer = len(self.backbone.encoder.layer)
        for prm in self.backbone.parameters():
            prm.requires_grad = False
        for blk in self.backbone.encoder.layer[max(0, n_layer - unfreeze_last):]:
            for prm in blk.parameters():
                prm.requires_grad = True
        for prm in self.backbone.layernorm.parameters():
            prm.requires_grad = True
        # v21: patch embedding ikut di-unfreeze -- triplet RGB genuine (3 slice fisik berbeda)
        # butuh layer paling awal bisa belajar interpretasi channel (pelajaran v16).
        for prm in self.backbone.embeddings.patch_embeddings.parameters():
            prm.requires_grad = True
        dim = self.backbone.config.hidden_size
        self.head = AnatomyQueryHead(dim * 2, d=256, n_target=len(LABEL_COLS), n_plane=len(PLANES))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"backbone: {n_layer} blok, {unfreeze_last} terakhir + layernorm + patch_emb trainable, "
              f"dim={dim}, total trainable params={n_trainable/1e6:.1f}M")

    def forward(self, imgs, mask, plane_id, position):
        B, T, C, H, W = imgs.shape
        x = imgs.reshape(B * T, C, H, W)               # sudah /255 + triplet RGB
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state   # (B*T, tokens, dim)
        cls = out[:, 0]
        patch_mean = out[:, 1:].mean(1)
        feat = torch.cat([cls, patch_mean], dim=1)     # (B*T, 2*dim)
        feat = feat.reshape(B, T, -1)
        return self.head(feat, mask, plane_id, position)

## 6. Sanity check arsitektur (tensor dummy)

In [ ]:
import math
_dummy_imgs = torch.rand(2, N_TOKEN, 3, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
_dummy_mask = torch.ones(2, N_TOKEN); _dummy_mask[1, -1] = 0.0
_dummy_plane = torch.zeros(2, N_TOKEN, dtype=torch.int64)
_dummy_pos = torch.linspace(0, 1, N_TOKEN).unsqueeze(0).expand(2, -1)
try:
    _m = AnatomyModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
    with torch.no_grad():
        _out = _m(_dummy_imgs.to(DEVICE), _dummy_mask.to(DEVICE),
                  _dummy_plane.to(DEVICE), _dummy_pos.to(DEVICE))
    assert _out.shape == (2, len(LABEL_COLS)), f"shape salah: {_out.shape}"
    tlog(f"Sanity check OK. Output shape: {tuple(_out.shape)}")
    del _m, _out
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
except Exception as e:
    tlog(f"SANITY CHECK GAGAL: {e}")
    raise

## 7. Split train/val (k-fold, gold dibagi 5 grup)

In [ ]:
GOLD_STUDIES = train_labels.loc[train_labels["source"] == "gold", "StudyInstanceUID"].tolist()
WEAK_STUDIES = train_labels.loc[train_labels["source"] == "flight0234_v4hybrid", "StudyInstanceUID"].tolist()

GOLD_STUDIES = [s for s in GOLD_STUDIES if s in {k[0] for k in study_plane_cache}]
WEAK_STUDIES = [s for s in WEAK_STUDIES if s in {k[0] for k in study_plane_cache}]
tlog(f"Studi gold (k-fold): {len(GOLD_STUDIES)} | studi label-LLM (training semua fold): {len(WEAK_STUDIES)}")
assert len(WEAK_STUDIES) > 0, "WEAK_STUDIES kosong"
assert len(GOLD_STUDIES) > 0, "GOLD_STUDIES kosong"

K_FOLDS = 5
_fold_rng = np.random.RandomState(20260819)
_gold_shuffled = list(GOLD_STUDIES)
_fold_rng.shuffle(_gold_shuffled)
FOLD_VAL_IDS = [_gold_shuffled[i::K_FOLDS] for i in range(K_FOLDS)]
for _i, _f in enumerate(FOLD_VAL_IDS):
    tlog(f"  fold {_i}: {len(_f)} studi gold validasi")
assert sum(len(f) for f in FOLD_VAL_IDS) == len(GOLD_STUDIES), "fold assignment tidak menutupi semua studi gold"

## 7b. Diagnostik (murah, sebelum training)

In [ ]:
def dataset_stats(ds, name, n_sample=200):
    n = min(n_sample, len(ds))
    idxs = np.random.RandomState(0).choice(len(ds), n, replace=False)
    mask_sums, target_valid_counts, img_means = [], [], []
    for i in idxs:
        item = ds[i]
        mask_sums.append(item["mask"].sum().item())
        target_valid_counts.append((~torch.isnan(item["target"])).sum().item())
        img_means.append(item["imgs"].mean().item())
    mask_sums = np.array(mask_sums); tvc = np.array(target_valid_counts); img_means = np.array(img_means)
    print(f"--- {name} (n={n}) ---")
    print(f"  mask.sum() per studi: mean={mask_sums.mean():.2f} min={mask_sums.min():.0f} max={mask_sums.max():.0f} "
          f"| studi dgn mask KOSONG: {(mask_sums==0).sum()} ({(mask_sums==0).mean()*100:.1f}%)")
    print(f"  label valid per studi: mean={tvc.mean():.2f} | studi TANPA label: {(tvc==0).sum()} ({(tvc==0).mean()*100:.1f}%)")
    print(f"  rata2 pixel image: mean={img_means.mean():.2f} | studi dgn gambar KOSONG: {(img_means < 1.0).sum()} ({(img_means<1.0).mean()*100:.1f}%)")
    return float((img_means < 1.0).mean())

_diag_train_ds = AnatomyDataset(WEAK_STUDIES, train_labels)
_diag_val_ds = AnatomyDataset(GOLD_STUDIES, train_labels)
_train_blank_frac = dataset_stats(_diag_train_ds, "train_ds (semua weak-label)")
_val_blank_frac = dataset_stats(_diag_val_ds, "val_ds (semua gold)")
DATA_OK = (_train_blank_frac < 0.5) and (_val_blank_frac < 0.5)
tlog(f"DATA_OK = {DATA_OK} (fraksi gambar kosong: train={_train_blank_frac*100:.1f}%, val={_val_blank_frac*100:.1f}%)")


In [ ]:
# 2) Variasi embedding antar-studi -- forward pass kecil, TANPA training.
_diag_model = AnatomyModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
_diag_model.eval()
_diag_loader = DataLoader(_diag_train_ds, batch_size=4, shuffle=True, num_workers=2)
_feats_list, _preds_list, _targets_list = [], [], []
with torch.no_grad():
    for bi, batch in enumerate(_diag_loader):
        if bi >= 2:
            break
        imgs = batch["imgs"].to(DEVICE); mask = batch["mask"].to(DEVICE)
        plane_id = batch["plane_id"].to(DEVICE); position = batch["position"].to(DEVICE)
        target = batch["target"].to(DEVICE)
        B, T, C, H, W = imgs.shape
        x = imgs.reshape(B * T, C, H, W)
        x = (x - _diag_model.mean) / _diag_model.std
        out = _diag_model.backbone(pixel_values=x).last_hidden_state
        cls = out[:, 0]; patch_mean = out[:, 1:].mean(1)
        feat = torch.cat([cls, patch_mean], dim=1).reshape(B, T, -1)   # (B,T,2*dim)
        logits = _diag_model.head(feat, mask, plane_id, position)
        _feats_list.append(feat.mean(1).cpu())
        _preds_list.append(torch.sigmoid(logits).cpu())
        _targets_list.append(target.cpu())

FEATS = torch.cat(_feats_list)
PREDS = torch.cat(_preds_list).numpy()
TARGETS_DIAG = torch.cat(_targets_list).numpy()

pairwise_std = FEATS.std(dim=0).mean().item()
print(f"Variasi embedding antar-studi (rata2 std per dimensi): {pairwise_std:.4f}")
print(f"\nRentang prediksi model (setelah sigmoid) per label:")
print(pd.DataFrame(PREDS, columns=LABEL_COLS).describe().loc[["mean","std","min","max"]].T)

del _diag_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 8. Loss & training loop (k-fold, identik v18/v20)

In [ ]:
def masked_bce(logits, target, weight=None):
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return logits.sum() * 0.0
    t = torch.where(mask, target, torch.zeros_like(target))
    loss = F.binary_cross_entropy_with_logits(logits, t, reduction="none")
    w = mask.float()
    if weight is not None:
        w = w * torch.where(mask, weight, torch.zeros_like(weight))
    return (loss * w).sum() / w.sum().clamp_min(1e-6)

def macro_auc(y_true, y_pred):
    from sklearn.metrics import roc_auc_score
    scores = []
    for j in range(y_true.shape[1]):
        col_true, col_pred = y_true[:, j], y_pred[:, j]
        valid = ~np.isnan(col_true)
        if valid.sum() < 2 or len(np.unique(col_true[valid])) < 2:
            continue
        scores.append(roc_auc_score(col_true[valid], col_pred[valid]))
    return float(np.mean(scores)) if scores else float("nan"), scores

EPOCHS = 8
BATCH_STUDIES = 4
LR_HEAD = 1e-3
LR_BACKBONE = 8e-6
SAFETY_CUTOFF_SECONDS = 7.0 * 3600
RUN_FULL_TRAINING = True

fold_results = []
oof_pred, oof_true = {}, {}

if not RUN_FULL_TRAINING:
    tlog("RUN_FULL_TRAINING=False -- lewati training loop, ini cuma push diagnostik.")
elif not DATA_OK:
    tlog("DATA_OK=False -- gambar/mask masih kosong massal, training DIBATALKAN.")
else:
    for fold_idx in range(K_FOLDS):
        if time.time() - T0 > SAFETY_CUTOFF_SECONDS:
            tlog(f"BATAS WAKTU AMAN TERCAPAI -- berhenti sebelum fold {fold_idx} mulai.")
            break
        tlog(f"===== FOLD {fold_idx+1}/{K_FOLDS} =====")

        val_ids_f = FOLD_VAL_IDS[fold_idx]
        val_id_set_f = set(val_ids_f)
        extra_gold_train_f = [s for s in GOLD_STUDIES if s not in val_id_set_f]
        train_ids_f = WEAK_STUDIES + extra_gold_train_f
        train_ds_f = AnatomyDataset(train_ids_f, train_labels)
        val_ds_f = AnatomyDataset(val_ids_f, train_labels)
        tlog(f"  train_ds: {len(train_ds_f)} ({len(WEAK_STUDIES)} weak + {len(extra_gold_train_f)} gold) "
             f"| val_ds (OOF): {len(val_ds_f)}")

        model = AnatomyModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
        backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        opt = torch.optim.AdamW([
            {"params": head_params, "lr": LR_HEAD},
            {"params": backbone_params, "lr": LR_BACKBONE},
        ], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS))
        train_loader = DataLoader(train_ds_f, batch_size=BATCH_STUDIES, shuffle=True, num_workers=2, drop_last=True)
        val_loader = DataLoader(val_ds_f, batch_size=BATCH_STUDIES, shuffle=False, num_workers=2)

        best_auc_f = -1.0
        best_state_f, best_val_true_f, best_val_pred_f = None, None, None
        history_f = []
        for epoch in range(EPOCHS):
            if time.time() - T0 > SAFETY_CUTOFF_SECONDS:
                tlog(f"  BATAS WAKTU AMAN TERCAPAI di tengah fold {fold_idx}.")
                break
            model.train()
            running = 0.0
            for bi, batch in enumerate(train_loader):
                imgs = batch["imgs"].to(DEVICE); mask = batch["mask"].to(DEVICE)
                plane_id = batch["plane_id"].to(DEVICE); position = batch["position"].to(DEVICE)
                target = batch["target"].to(DEVICE); weight = batch["weight"].to(DEVICE)
                logits = model(imgs, mask, plane_id, position)
                loss = masked_bce(logits, target, weight)
                opt.zero_grad(); loss.backward(); opt.step()
                if XLA_OK:
                    xm.mark_step()
                running += loss.item()
                if (bi + 1) % 100 == 0:
                    tlog(f"  fold {fold_idx+1} epoch {epoch+1} batch {bi+1}/{len(train_loader)} loss={running/(bi+1):.4f}")

            model.eval()
            all_true, all_pred = [], []
            with torch.no_grad():
                for batch in val_loader:
                    imgs = batch["imgs"].to(DEVICE); mask = batch["mask"].to(DEVICE)
                    plane_id = batch["plane_id"].to(DEVICE); position = batch["position"].to(DEVICE)
                    target = batch["target"].to(DEVICE)
                    logits = model(imgs, mask, plane_id, position)
                    all_true.append(target.cpu().numpy())
                    all_pred.append(torch.sigmoid(logits).cpu().numpy())
            y_true = np.concatenate(all_true) if all_true else np.zeros((0, len(LABEL_COLS)))
            y_pred = np.concatenate(all_pred) if all_pred else np.zeros((0, len(LABEL_COLS)))
            auc, per_label = macro_auc(y_true, y_pred)
            cur_lr_head = opt.param_groups[0]["lr"]
            cur_lr_backbone = opt.param_groups[1]["lr"]
            history_f.append({"fold": fold_idx, "epoch": epoch + 1, "train_loss": running / max(1, len(train_loader)),
                               "val_macro_auc": auc, "lr_head": cur_lr_head, "lr_backbone": cur_lr_backbone})
            tlog(f"  FOLD {fold_idx+1} EPOCH {epoch+1}/{EPOCHS} selesai. train_loss={running/max(1,len(train_loader)):.4f} "
                 f"val_macro_auc={auc:.4f} lr_head={cur_lr_head:.2e} lr_backbone={cur_lr_backbone:.2e}")

            if auc == auc and auc > best_auc_f:
                best_auc_f = auc
                best_state_f = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_val_true_f, best_val_pred_f = y_true, y_pred
                tlog(f"    -> checkpoint fold {fold_idx+1} disimpan (val_macro_auc={auc:.4f})")

            scheduler.step()

        if best_state_f is not None:
            torch.save(best_state_f, OUT_DIR / f"best_model_fold{fold_idx}.pt")
            for sid, pred_row, true_row in zip(val_ds_f.study_ids, best_val_pred_f, best_val_true_f):
                oof_pred[sid] = pred_row
                oof_true[sid] = true_row

        fold_results.append({"fold": fold_idx, "best_val_macro_auc": best_auc_f, "n_val": len(val_ds_f)})
        pd.DataFrame(history_f).to_csv(OUT_DIR / f"training_history_fold{fold_idx}.csv", index=False)
        tlog(f"FOLD {fold_idx+1}/{K_FOLDS} SELESAI. Best val macro AUC: {best_auc_f:.4f}")

        del model, opt, scheduler
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    pd.DataFrame(fold_results).to_csv(OUT_DIR / "fold_results.csv", index=False)
    fold_aucs = [r["best_val_macro_auc"] for r in fold_results if r["best_val_macro_auc"] == r["best_val_macro_auc"]]
    if fold_aucs:
        tlog(f"Rata-rata AUC per-fold: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f} "
             f"(min={min(fold_aucs):.4f}, max={max(fold_aucs):.4f}, n_fold={len(fold_aucs)})")

    if oof_pred:
        oof_ids = list(oof_pred.keys())
        oof_y_true = np.stack([oof_true[s] for s in oof_ids])
        oof_y_pred = np.stack([oof_pred[s] for s in oof_ids])
        oof_auc, oof_per_label = macro_auc(oof_y_true, oof_y_pred)
        tlog(f"OOF MACRO AUC (gabungan {len(oof_ids)}/{len(GOLD_STUDIES)} studi gold): {oof_auc:.4f}")
        oof_df = pd.DataFrame({"StudyInstanceUID": oof_ids})
        for i, c in enumerate(LABEL_COLS):
            oof_df[f"pred_{c}"] = oof_y_pred[:, i]
            oof_df[f"true_{c}"] = oof_y_true[:, i]
        oof_df.to_csv(OUT_DIR / "oof_predictions.csv", index=False)
        tlog(f"Perbandingan: v15 (1 split) = 0.8304 | v18 OOF (SlotHead, cache v2) = 0.8068 | v21 OOF (AnatomyQuery, cache v4) = {oof_auc:.4f}")
    else:
        tlog("[warn] tidak ada prediksi OOF tersimpan (kemungkinan SAFETY_CUTOFF_SECONDS tercapai).")

## Catatan v21

- **Perubahan TUNGGAL dari v20**: head (SlotHead -> AnatomyQueryHead) + dataset (6-slot bag -> 24
  token anatomis, 8 per plane). Semua training hyperparameter, k-fold, loss, scheduler identik.
- **Pembanding**: v18 OOF (SlotHead, cache v2) = 0,8068; v20 OOF (SlotHead, cache v4) = belum jalan.
- **RGB triplet + patch_embed di-unfreeze** (pelajaran v16): triplet genuine 3 slice berdekatan,
  layer paling awal dibuat trainable supaya bisa belajar interpretasi channel.
- **Sagittal reversal utk lutut kiri** (resep prvsiyan V19): token index 0 = paling medial selalu.
- Kalau OOF v21 > v18/v20 melebihi noise antar-fold (±0,034): anatomy-query head = pengungkit nyata.
  5 checkpoint tersimpan (best_model_fold0-4.pt) kandidat ensemble 5-model.